# EXTRACTOR
## TYPE: EXAM GENIUS (Daily to weekly pdf's)
### STATUS: OK

<div style="text-align: center;">
    <div style="display: inline-block; margin-right: 10px;">
        <img src="Capture1.JPG" alt="Capture1" width="300"/>
    </div>
    <div style="display: inline-block;">
        <img src="Capture.JPG" alt="Capture" width="300"/>
    </div>
</div>

In [ ]:
import fitz
import re
import pandas as pd

def clean_text(text):
    text = re.split(r"/", text)[0]
    text = re.sub(r'[\u0900-\u097F]+', '', text)
    text = re.sub(r'[-–,.;:]*\s*\d*\s*$', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_mcqs_from_pdf(pdf_path, output_file="mcqs.csv"):
    questions = []
    doc = fitz.open(pdf_path)

    for page in doc:
        text = page.get_text("text")

        blocks = re.split(r"\n(?=Ques:)", text)

        for block in blocks:
            if block.strip().startswith("Ques:"):
                lines = block.strip().split("\n")

                question_lines = []
                options = {}
                correct = ""

                for line in lines:
                    if line.startswith("Ques:"):
                        question_lines.append(clean_text(line.replace("Ques:", "").strip()))
                    elif re.match(r"^[A-E]\.", line.strip()):
                        key = line[0]
                        options[key] = clean_text(line[2:].strip())
                    elif line.startswith("Answer:"):
                        correct = clean_text(line.replace("Answer:", "").strip())
                    else:
                        if not options and not line.startswith("Explanation:"):
                            question_lines.append(clean_text(line.strip()))

                full_question = " ".join(question_lines).strip()

                correct_answer_text = ""
                match = re.search(r"Option\s+([A-E])", correct, re.IGNORECASE)
                if match:
                    option_key = match.group(1).upper()
                    correct_answer_text = options.get(option_key, "")

                questions.append({
                    "Question": full_question,
                    "Option1": options.get("A", ""),
                    "Option2": options.get("B", ""),
                    "Option3": options.get("C", ""),
                    "Option4": options.get("D", ""),
                    "Option5": options.get("E", ""),
                    "Answer": correct_answer_text
                })

    df = pd.DataFrame(questions)
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"✅ Extracted {len(df)} questions and saved to {output_file}")

extract_mcqs_from_pdf("Test_extract.pdf")

## TYPE: Monthly pdf's
### STATUS: TBM

In [ ]:
import fitz
import re
import pandas as pd
import csv
import os

def clean_text(text):
    text = re.sub(r'[\u0900-\u097F]+', '', text)  # Remove Hindi
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_option(opt):
    return re.sub(r"^[\(]?[a-eA-E][\.\)\-]?\s*", "", opt).strip()

def remove_boilerplate(text):
    junk_patterns = [
        r"No\.?\s*1\s*PLATFORM\s*FOR\s*GOVT\.?\s*EXAMS",
        r"UPSC\s*\|\s*SSC\s*\|\s*BANKING\s*\|\s*RAILWAY",
        r"www\.examgenius\.in",
        r"Visit our website.*?",
        r"Exam[_\s]?Genius[_\s]?Official",
        r"Exam Genius",
        r"Visit",
    ]
    for pat in junk_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    return text

def extract_mcqs_from_pdf(pdf_path, output_file="mcqs.csv"):
    questions = []
    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc):
        if page_num == 0:
            continue

        text = page.get_text("text")
        text = remove_boilerplate(text)

        if not re.search(r"Ques", text, re.IGNORECASE):
            continue

        blocks = re.split(r"(?:\n|\r)?Ques\s*[:\-]", text, flags=re.IGNORECASE)
        for block in blocks:
            block = block.strip()
            if not block or len(block) < 20:
                continue

            q_match = re.split(r"\([aA]\)|\(a\)", block, maxsplit=1)
            question = clean_text(q_match[0])

            option_matches = re.findall(r"\([a-eA-E]\)\s*([^\n]+)", block)
            options = [clean_option(opt) for opt in option_matches]

            while len(options) < 5:
                options.append("")

            ans_match = re.search(r"Answer\s*[:\-]?\s*Option\s*([A-E])", block, re.IGNORECASE)
            correct_letter = ans_match.group(1).upper() if ans_match else ""

            correct_answer = ""
            if correct_letter:
                idx = ord(correct_letter) - 65
                if 0 <= idx < len(options):
                    correct_answer = options[idx]

            if len(question.split()) < 3 or all(not opt for opt in options):
                continue

            questions.append({
                "Question": question,
                "Option1": options[0],
                "Option2": options[1],
                "Option3": options[2],
                "Option4": options[3],
                "Option5": options[4],
                "Answer": correct_answer
            })

    df = pd.DataFrame(questions, columns=["Question", "Option1", "Option2", "Option3", "Option4", "Option5", "Answer"])
    df.to_csv(output_file, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
    print(f"✅ Extracted {len(df)} MCQs and saved to '{output_file}'.")
    return len(df)


pdf_files = [f for f in os.listdir('.') if f.lower().endswith('.pdf')]
if not pdf_files:
    print("❌ No PDF files found in this folder.")
    exit()

print("\n📂 Available PDF files:\n")
for i, pdf in enumerate(pdf_files, start=1):
    print(f"{i}. {pdf}")

choice = input("\n👉 Enter the number of the PDF you want to process: ")
try:
    pdf_path = pdf_files[int(choice) - 1]
except (ValueError, IndexError):
    print("❌ Invalid choice.")
    exit()

total_questions = extract_mcqs_from_pdf(pdf_path, output_file="temp.csv")

base_name = os.path.splitext(os.path.basename(pdf_path))[0]
new_csv_name = f"{base_name} ({total_questions}).csv"

os.rename("temp.csv", new_csv_name)
print(f"📁 Final CSV saved as: {new_csv_name}")

## TYPE: Adda 247
### STATUS: OK

<div style="text-align: center;">
    <div style="display: inline-block; margin-right: 10px;">
        <img src="Capture2.JPG" alt="Capture2" width="300"/>
    </div>
    <div style="display: inline-block;">
        <img src="Capture3.JPG" alt="Capture3" width="300"/>
    </div>
</div>

In [ ]:
import fitz
import re
import pandas as pd

def clean_text(text):
    text = re.split(r"/", text)[0]
    text = re.sub(r'[\u0900-\u097F]+', '', text)
    text = re.sub(r'[-–,.;:]*\s*\d*\s*$', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def normalize_question(text):
    """Prepare text for duplicate detection (case + punctuation insensitive)."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]+', '', text)
    return text.strip()

def extract_mcqs_from_pdf(pdf_path, output_file="mcqs.xlsx"):
    questions = []
    doc = fitz.open(pdf_path)

    text = ""
    for page in doc:
        text += page.get_text("text") + "\n"

    blocks = re.split(r"\n?Q\d+\.", text)

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        lines = block.split("\n")
        question_lines = []
        options = {}
        correct = ""

        for line in lines:
            line = line.strip()
            if re.match(r"^\([a-eA-E]\)", line):
                key = line[1].upper()
                options[key] = clean_text(line[3:].strip())
            elif line.startswith("Answer:"):
                correct = clean_text(line.replace("Answer:", "").strip())
            else:
                question_lines.append(clean_text(line))

        full_question = " ".join(question_lines).strip()

        correct_answer_text = ""
        match = re.search(r"\(?([a-eA-E])\)?", correct)
        if match:
            option_key = match.group(1).upper()
            correct_answer_text = options.get(option_key, "")

        questions.append({
            "Question": full_question,
            "NormQ": normalize_question(full_question),
            "Option1": options.get("A", ""),
            "Option2": options.get("B", ""),
            "Option3": options.get("C", ""),
            "Option4": options.get("D", ""),
#             "Option5": options.get("E", ""),
            "Answer": correct_answer_text
        })

    df = pd.DataFrame(questions)

    df = df.drop_duplicates(subset=["NormQ"]).drop(columns=["NormQ"])

    df.to_excel(output_file, index=False)
    print(f"✅ Extracted {len(df)} unique questions and saved to {output_file}")

def fill_answers_from_solutions(pdf_path, excel_path, output_path):
    df = pd.read_excel(excel_path)

    doc = fitz.open(pdf_path)
    answers_text = ""
    for i in range(64, len(doc)):
        answers_text += doc[i].get_text("text") + "\n"

    solutions_part = answers_text.split("Solutions", 1)[-1]

    answer_map = {}
    for match in re.finditer(r"S(\d+)\.\s*Ans\.\((\w)\)", solutions_part):
        qnum = int(match.group(1))
        ans_letter = match.group(2).upper()
        answer_map[qnum] = ans_letter

    df_filled = df.copy()

    for idx in df_filled.index:
        qnum = idx + 1
        if qnum in answer_map:
            ans_letter = answer_map[qnum]
            option_col = f"Option{ord(ans_letter) - 64}"
            if option_col in df_filled.columns:
                df_filled.at[idx, "Answer"] = df_filled.at[idx, option_col]

    df_filled.to_excel(output_path, index=False)
    print(f"✅ Answers filled and saved to {output_path}")

if __name__ == "__main__":
    extract_mcqs_from_pdf("Test_extract.pdf", "mcqs_unique.xlsx")

    fill_answers_from_solutions(
        pdf_path="Adda247 1000+ Qs.pdf",
        excel_path="mcqs_unique.xlsx",
        output_path="Master_MCQs_with_Answers.xlsx"
    )

# CLEANER (Duplicate Remover)
### STATUS: OK

In [ ]:
import os
import pandas as pd

output_file = "quiz_data.csv"

df = pd.read_csv(output_file)

original_count = len(df)
print(f"\n📊 Total questions in merged file: {original_count}")

duplicates = df[df.duplicated(subset=["Question"], keep=False)]

if not duplicates.empty:
    print("\n❌ Duplicate questions found:")
    for question, group in duplicates.groupby("Question"):
        indices = group.index.tolist()
        print(f" - '{question}' found at rows: {indices}")
else:
    print("\n✅ No duplicate questions found!")

clean_df = df.drop_duplicates(subset=["Question"], keep='first')

new_count = len(clean_df)
print(f"\n✅ Total questions after removing duplicates: {new_count}")
print(f"🧹 Total duplicates removed: {original_count - new_count}")

clean_df.to_csv(output_file, index=False)
print(f"\n💾 Cleaned and merged CSV saved as '{output_file}' (duplicates removed).")

# INDEXER
## TYPE: Exam Genius
### STATUS: OK

In [ ]:
import pandas as pd

input_file = "quiz_data.csv"
output_file = "updated_data.csv"

df = pd.read_csv(input_file)

df.insert(0, "Q no", range(2, len(df) + 2))

df.to_csv(output_file, index=False)

print(f"New CSV with Question numbers saved as '{output_file}'")

## TYPE: Adda247
### STATUS: OK

In [ ]:
import pandas as pd

input_file = "quiz_data_adda247.csv"
output_file = "updated_data_adda247.csv"

df = pd.read_csv(input_file)

df.insert(0, "Q No", range(1, len(df) + 1))

df["Option5"] = ""

desired_order = ["Q No", "Question", "Option1", "Option2", "Option3", "Option4", "Option5", "Answer"]
df = df[desired_order]

df.to_csv(output_file, index=False)

print(f"✅ New CSV with 'Q No' and empty 'Option5' saved as '{output_file}'")

# MERGER
### STATUS: OK

In [ ]:
import os
import pandas as pd

input_folder = "./csvs"
output_file = "quiz_data.csv"

columns = ["Question", "Option1", "Option2", "Option3", "Option4", "Option5", "Answer"]

all_data = []

for file_name in os.listdir(input_folder):
    if file_name.endswith(".csv"):
        file_path = os.path.join(input_folder, file_name)
        df = pd.read_csv(file_path)
        
        df = df[columns]
        all_data.append(df)

merged_df = pd.concat(all_data, ignore_index=True)
merged_df.to_csv(output_file, index=False)

print(f"✅ Merged {len(all_data)} CSV files into '{output_file}' successfully!")

# MANAGER
### STATUS: OK

In [2]:
import tkinter as tk
from tkinter import ttk, filedialog, Toplevel, messagebox
import pandas as pd
import os
import re

CSV_FILE = "dummy.csv"
COLUMNS = ["Q no", "Question", "Option1", "Option2", "Option3", "Option4", "Option5", "Answer"]

def load_data(csv_file=CSV_FILE):
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file, dtype=str)
        df.columns = df.columns.str.strip()
        for c in COLUMNS:
            if c not in df.columns:
                df[c] = ""
        df = df.reindex(columns=COLUMNS)
        if "Q no" not in df.columns or df["Q no"].isnull().any():
            df["Q no"] = range(1, len(df) + 1)
        try:
            df["Q no"] = df["Q no"].astype(int)
        except Exception:
            df["Q no"] = range(1, len(df) + 1)
        return df
    return pd.DataFrame(columns=COLUMNS)

def save_data(df, csv_file=CSV_FILE):
    df = df.reindex(columns=COLUMNS)
    df["Q no"] = range(1, len(df) + 1)
    df.to_csv(csv_file, index=False)

class QuizManager:
    def __init__(self, root):
        self.root = root
        self.root.title("Quiz Manager")
        self.root.geometry("1200x700")
        self.df = load_data()
        self.edit_popup = None
        self.filtered_indices = []
        self.popup_mode = "all"
        self.current_csv_file = CSV_FILE

        search_frame = tk.Frame(root, bg="#ecf0f1", pady=5)
        search_frame.pack(fill="x")
        tk.Label(search_frame, text="Search:", font=("Segoe UI", 10), bg="#ecf0f1").pack(side="left", padx=5)
        self.search_var = tk.StringVar()
        search_entry = tk.Entry(search_frame, textvariable=self.search_var, font=("Segoe UI", 10))
        search_entry.pack(side="left", fill="x", expand=True, padx=5)
        search_entry.bind("<Return>", lambda event: self.search_questions())
        tk.Button(search_frame, text="Search", command=self.search_questions).pack(side="left", padx=5)
        tk.Button(search_frame, text="Clear", command=self.clear_search).pack(side="left", padx=5)

        self.sidebar = tk.Frame(root, bg="#2c3e50", width=220)
        self.sidebar.pack(side="left", fill="y")
        self.content = tk.Frame(root, bg="#ecf0f1")
        self.content.pack(side="right", fill="both", expand=True)

        style = ttk.Style()
        style.theme_use("clam")
        style.configure("Treeview.Heading", font=("Segoe UI", 10, "bold"), foreground="#333")
        style.configure("Treeview", font=("Arial", 10), rowheight=25, background="#f9f9f9", fieldbackground="#f9f9f9")
        style.map("Treeview", background=[("selected", "#6fa1f2")], foreground=[("selected", "white")])

        tk.Label(self.sidebar, text="Select CSV:", bg="#2c3e50", fg="white", font=("Segoe UI", 10, "bold")).pack(anchor="w", padx=10, pady=(10, 0))
        self.csv_var = tk.StringVar()
        self.csv_dropdown = ttk.Combobox(self.sidebar, textvariable=self.csv_var, state="readonly", width=25)
        self.csv_dropdown.pack(padx=10, pady=5)
        self.refresh_csv_list()
        self.csv_dropdown.bind("<<ComboboxSelected>>", self.load_selected_csv)

        btn_specs = [
            ("Preview All", self.refresh_table),
            ("Add New Question", self.open_add_window),
            ("Copy Question(s)", self.copy_selected_questions),
            ("Edit Question(s)", self.open_edit_questions_popup),
            ("Delete Question", self.delete_question),
            ("Import CSV", self.import_csv),
            ("Export CSV", self.export_csv),
            ("Remove Duplicates", self.remove_duplicates),
            ("Save Changes", self.save_current_csv)
        ]
        for text, cmd in btn_specs:
            b = tk.Button(self.sidebar, text=text, command=cmd, bg="#34495e", fg="white",
                          relief="raised", activebackground="#1abc9c", activeforeground="white",
                          font=("Segoe UI", 10), padx=10, pady=5, anchor="w")
            b.pack(pady=5, fill="x")

        self.counter = tk.Label(self.sidebar, text=f"Total Questions: {len(self.df)}", bg="#2c3e50", fg="white", font=("Segoe UI", 10))
        self.counter.pack(pady=20, padx=10, anchor="w")

        self.table_frame = tk.Frame(self.content)
        self.table_frame.pack(fill="both", expand=True, padx=10, pady=10)
        self.tree = ttk.Treeview(self.table_frame, columns=COLUMNS, show="headings", selectmode="extended")
        vsb = ttk.Scrollbar(self.table_frame, orient="vertical", command=self.tree.yview)
        hsb = ttk.Scrollbar(self.table_frame, orient="horizontal", command=self.tree.xview)
        self.tree.configure(yscroll=vsb.set, xscroll=hsb.set)
        self.tree.grid(row=0, column=0, sticky="nsew")
        vsb.grid(row=0, column=1, sticky="ns")
        hsb.grid(row=1, column=0, sticky="ew")
        self.table_frame.grid_rowconfigure(0, weight=1)
        self.table_frame.grid_columnconfigure(0, weight=1)

        for col in COLUMNS:
            self.tree.heading(col, text=col)
            if col == "Q no":
                self.tree.column(col, width=50, anchor="center", stretch=False)
            elif col == "Question":
                self.tree.column(col, width=400, anchor="w")
            else:
                self.tree.column(col, width=150, anchor="w")

        self.tree.bind("<<TreeviewSelect>>", lambda e: self.view_selected_question())
        self.tree.bind("<Up>", lambda e: self.view_selected_question())
        self.tree.bind("<Down>", lambda e: self.view_selected_question())

        self.detail_frame = tk.LabelFrame(self.content, text="Selected Question Details", padx=10, pady=10)
        self.detail_frame.pack(fill="x", padx=10, pady=5)
        self.detail_text = tk.Text(self.detail_frame, height=10, wrap="word", state="disabled")
        self.detail_text.pack(fill="x")

        self.refresh_table()

    def refresh_csv_list(self):
        files = [f for f in os.listdir(".") if f.lower().endswith(".csv")]
        self.csv_dropdown['values'] = files
        if self.current_csv_file in files:
            self.csv_var.set(self.current_csv_file)
        elif files:
            self.csv_var.set(files[0])

    def load_selected_csv(self, event=None):
        file_path = self.csv_var.get()
        if not file_path:
            return
        try:
            self.df = load_data(file_path).fillna("")
            self.df.reset_index(drop=True, inplace=True)
            self.current_csv_file = file_path
            self.refresh_table()
            messagebox.showinfo("Loaded", f"CSV loaded successfully:\n{file_path}")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to load CSV:\n{str(e)}")

    def refresh_table(self):
        self.df["Q no"] = range(1, len(self.df) + 1)
        for r in self.tree.get_children():
            self.tree.delete(r)
        for idx, row in self.df.iterrows():
            values = [row.get(c, "") if pd.notna(row.get(c, "")) else "" for c in COLUMNS]
            self.tree.insert("", "end", iid=str(idx), values=values)
        self.counter.config(text=f"Total Questions: {len(self.df)}")

    def clear_search(self):
        self.search_var.set("")
        self.refresh_table()

    def search_questions(self):
        query = self.search_var.get().strip().lower()
        if query == "":
            self.refresh_table()
            return
        mask = self.df.apply(lambda row: query in " ".join([str(x) for x in row.fillna("")]).lower(), axis=1)
        filtered_df = self.df[mask]
        for row in self.tree.get_children():
            self.tree.delete(row)
        for idx, row in filtered_df.iterrows():
            self.tree.insert("", "end", iid=str(idx), values=list(row.fillna("")))

    def view_selected_question(self):
        selected_item = self.tree.selection()
        if not selected_item:
            return
        try:
            idx_label = int(selected_item[0])
        except Exception:
            return
        if idx_label not in self.df.index:
            return
        row = self.df.loc[idx_label]
        details = f"Question:\n{row['Question']}\n\n"
        for i in range(1, 6):
            details += f"Option {i}: {row.get(f'Option{i}', '')}\n"
        details += f"\nAnswer: {row.get('Answer', '')}"
        self.detail_text.config(state="normal")
        self.detail_text.delete("1.0", "end")
        self.detail_text.insert("1.0", details)
        self.detail_text.config(state="disabled")

    def open_add_window(self):
        add_win = Toplevel(self.root)
        add_win.title("Add New Question")
        add_win.geometry("550x500")
        add_win.grab_set()

        form_frame = tk.Frame(add_win)
        form_frame.pack(padx=20, pady=20)

        tk.Label(form_frame, text="Question:").grid(row=0, column=0, sticky="w", pady=5)
        self.question_text = tk.Text(form_frame, width=50, height=5, wrap="word")
        self.question_text.grid(row=0, column=1, pady=5)

        self.add_entries = {}
        for i, col in enumerate(["Option1", "Option2", "Option3", "Option4", "Option5", "Answer"], start=1):
            tk.Label(form_frame, text=col + ":").grid(row=i, column=0, sticky="w", pady=5)
            entry = tk.Entry(form_frame, width=50)
            entry.grid(row=i, column=1, pady=5)
            self.add_entries[col] = entry

        button_frame = tk.Frame(add_win)
        button_frame.pack(pady=10)
        tk.Button(button_frame, text="Add & Close", command=lambda: self.add_question(add_win, True)).pack(side="left", padx=5)
        tk.Button(button_frame, text="Add Another", command=lambda: self.add_question(add_win, False)).pack(side="left", padx=5)
        tk.Button(button_frame, text="Cancel", command=add_win.destroy).pack(side="left", padx=5)

    def add_question(self, window, close=True):
        question = self.question_text.get("1.0", "end").strip()
        answer = self.add_entries["Answer"].get().strip()
        if question == "" or answer == "":
            messagebox.showwarning("Warning", "Question and Answer are required!")
            return
        new_row = {"Question": question, "Answer": answer}
        for col in ["Option1", "Option2", "Option3", "Option4", "Option5"]:
            new_row[col] = self.add_entries[col].get().strip()
        self.df = pd.concat([self.df, pd.DataFrame([new_row])], ignore_index=True)
        save_data(self.df, self.current_csv_file)
        self.refresh_table()
        messagebox.showinfo("Success", "Question added successfully!")
        if close:
            window.destroy()
        else:
            self.question_text.delete("1.0", "end")
            for entry in self.add_entries.values():
                entry.delete(0, "end")

    def open_edit_questions_popup(self):
        if self.edit_popup and tk.Toplevel.winfo_exists(self.edit_popup):
            try:
                self.edit_popup.lift()
            except Exception:
                pass
            return

        self.edit_popup = Toplevel(self.root)
        self.edit_popup.title("Edit Question(s)")
        self.edit_popup.geometry("920x720")
        self.edit_popup.grab_set()

        container = tk.Frame(self.edit_popup)
        container.pack(fill="both", expand=True, padx=10, pady=10)

        control_frame = tk.Frame(container)
        control_frame.pack(fill="x", pady=(0, 8))

        all_btn = tk.Button(control_frame, text="All Questions", width=15)
        miss_btn = tk.Button(control_frame, text="Missing Only", width=15)
        all_btn.pack(side="left", padx=(0, 8))
        miss_btn.pack(side="left", padx=(0, 8))

        def update_button_colors():
            if self.popup_mode == "all":
                all_btn.config(bg="green", fg="white")
                miss_btn.config(bg="#f0f0f0", fg="black")
            else:
                miss_btn.config(bg="green", fg="white")
                all_btn.config(bg="#f0f0f0", fg="black")

        search_frame = tk.Frame(control_frame)
        search_frame.pack(side="right")
        tk.Label(search_frame, text="Search (Enter):").pack(side="left", padx=(0, 5))
        popup_search_var = tk.StringVar()
        popup_search_entry = tk.Entry(search_frame, textvariable=popup_search_var, width=30)
        popup_search_entry.pack(side="left", padx=(0, 5))
        clear_search_btn = tk.Button(search_frame, text="Clear Search", command=lambda: clear_search())
        clear_search_btn.pack(side="left")

        tree_frame = tk.Frame(container)
        tree_frame.pack(fill="both", expand=True)

        popup_tree = ttk.Treeview(tree_frame, columns=COLUMNS, show="headings", selectmode="extended")
        vsb = ttk.Scrollbar(tree_frame, orient="vertical", command=popup_tree.yview)
        hsb = ttk.Scrollbar(tree_frame, orient="horizontal", command=popup_tree.xview)
        popup_tree.configure(yscroll=vsb.set, xscroll=hsb.set)
        popup_tree.grid(row=0, column=0, sticky="nsew")
        vsb.grid(row=0, column=1, sticky="ns")
        hsb.grid(row=1, column=0, sticky="ew")
        tree_frame.grid_rowconfigure(0, weight=1)
        tree_frame.grid_columnconfigure(0, weight=1)

        for col in COLUMNS:
            popup_tree.heading(col, text=col)
            popup_tree.column(col, width=150 if col != "Question" else 400, anchor="w")

        editor_frame = tk.LabelFrame(container, text="Editor", padx=10, pady=8)
        editor_frame.pack(fill="x", pady=(8, 0))
        tk.Label(editor_frame, text="Question:").grid(row=0, column=0, sticky="nw", pady=4)
        editor_question_text = tk.Text(editor_frame, width=80, height=6, wrap="word")
        editor_question_text.grid(row=0, column=1, pady=4, sticky="w")

        editor_entries = {}
        for i, col in enumerate(["Option1", "Option2", "Option3", "Option4", "Option5", "Answer"], start=1):
            tk.Label(editor_frame, text=col + ":").grid(row=i, column=0, sticky="w", pady=4)
            entry = tk.Entry(editor_frame, width=80)
            entry.grid(row=i, column=1, pady=4, sticky="w")
            editor_entries[col] = entry

        nav_frame = tk.Frame(container)
        nav_frame.pack(fill="x", pady=10)
        status_label = tk.Label(nav_frame, text="", anchor="w")
        status_label.pack(side="left", padx=6)
        current_pos = {"pos": 0}

        def refresh_popup_table():
            popup_tree.delete(*popup_tree.get_children())
            df_to_show = self.df.copy()
            if self.popup_mode == "missing":
                mask_missing = df_to_show[["Question", "Option1", "Option2", "Option3", "Option4", "Option5", "Answer"]].isnull().any(axis=1) | \
                               (df_to_show[["Question", "Option1", "Option2", "Option3", "Option4", "Option5", "Answer"]] == "").any(axis=1)
                df_to_show = df_to_show[mask_missing]
            query = popup_search_var.get().strip().lower()
            if query:
                mask = df_to_show.apply(lambda row: query in " ".join([str(x).lower() for x in row.values]), axis=1)
                df_to_show = df_to_show[mask]
            self.filtered_indices = df_to_show.index.tolist()
            for idx, row in df_to_show.iterrows():
                popup_tree.insert("", "end", iid=str(idx), values=list(row.fillna("")))
            if self.filtered_indices:
                if current_pos["pos"] >= len(self.filtered_indices):
                    current_pos["pos"] = len(self.filtered_indices) - 1
                popup_tree.selection_set(str(self.filtered_indices[current_pos["pos"]]))
                popup_tree.focus(str(self.filtered_indices[current_pos["pos"]]))
                load_current()
            else:
                clear_editor()
                status_label.config(text="No questions in current filter/search")

        def set_all_mode():
            self.popup_mode = "all"
            update_button_colors()
            refresh_popup_table()

        def set_missing_mode():
            self.popup_mode = "missing"
            update_button_colors()
            refresh_popup_table()

        all_btn.config(command=set_all_mode)
        miss_btn.config(command=set_missing_mode)
        update_button_colors()
        popup_search_entry.bind("<Return>", lambda e: refresh_popup_table())

        def clear_search():
            popup_search_var.set("")
            refresh_popup_table()

        def clear_editor():
            editor_question_text.delete("1.0", "end")
            for col in editor_entries:
                editor_entries[col].delete(0, "end")

        def load_current():
            if not self.filtered_indices:
                clear_editor()
                return
            pos = current_pos["pos"]
            if pos < 0 or pos >= len(self.filtered_indices):
                return
            idx_label = self.filtered_indices[pos]
            if idx_label not in self.df.index:
                clear_editor()
                return
            row = self.df.loc[idx_label]
            editor_question_text.delete("1.0", "end")
            editor_question_text.insert("1.0", row.get("Question", "") if pd.notna(row.get("Question", "")) else "")
            for col in editor_entries:
                editor_entries[col].delete(0, "end")
                val = row.get(col, "")
                if pd.notna(val):
                    editor_entries[col].insert(0, val)
            status_label.config(text=f"Editing {pos + 1}/{len(self.filtered_indices)} (Row #{idx_label + 1})")

        def save_current():
            if not self.filtered_indices:
                return
            idx_label = self.filtered_indices[current_pos["pos"]]
            self.df.at[idx_label, "Question"] = editor_question_text.get("1.0", "end").strip()
            for col in editor_entries:
                self.df.at[idx_label, col] = editor_entries[col].get().strip()
            save_data(self.df, self.current_csv_file)
            self.refresh_table()
            load_current()
            status_label.config(text=f"Saved (Row #{idx_label + 1})")

        def prev_action():
            if current_pos["pos"] > 0:
                save_current()
                current_pos["pos"] -= 1
                sel_label = self.filtered_indices[current_pos["pos"]]
                popup_tree.selection_set(str(sel_label))
                popup_tree.focus(str(sel_label))
                load_current()

        def next_action():
            if current_pos["pos"] < len(self.filtered_indices) - 1:
                save_current()
                current_pos["pos"] += 1
                sel_label = self.filtered_indices[current_pos["pos"]]
                popup_tree.selection_set(str(sel_label))
                popup_tree.focus(str(sel_label))
                load_current()

        def on_tree_click(event):
            item = popup_tree.focus()
            if not item:
                return
            try:
                idx_label = int(item)
            except Exception:
                return
            if idx_label in self.filtered_indices:
                current_pos["pos"] = self.filtered_indices.index(idx_label)
                load_current()

        def on_tree_select(event):
            sel = popup_tree.selection()
            if not sel:
                return
            try:
                idx_label = int(sel[0])
            except Exception:
                return
            if idx_label in self.filtered_indices:
                current_pos["pos"] = self.filtered_indices.index(idx_label)
                load_current()

        sentence_end_re = re.compile(r'[.!?]\s+')

        def text_move_to_sentence_start(text_widget):
            """Move insert to start of current sentence in a Text widget."""
            full = text_widget.get("1.0", "end-1c")
            before = text_widget.get("1.0", "insert")
            pos = len(before)
            last_end = None
            for m in sentence_end_re.finditer(full):
                if m.end() <= pos:
                    last_end = m.end()
                else:
                    break
            new_pos = last_end if last_end is not None else 0
            text_widget.mark_set("insert", f"1.0+{new_pos}c")
            text_widget.see("insert")

        def text_move_to_sentence_end(text_widget):
            """Move insert to end of current sentence in a Text widget."""
            full = text_widget.get("1.0", "end-1c")
            before = text_widget.get("1.0", "insert")
            pos = len(before)
            found = None
            for m in sentence_end_re.finditer(full):
                if m.start() >= pos:
                    found = m.end()
                    break
            if found is None:
                new_pos = len(full)
            else:
                new_pos = found
            text_widget.mark_set("insert", f"1.0+{new_pos}c")
            text_widget.see("insert")

        def entry_move_to_sentence_start(entry_widget):
            s = entry_widget.get()
            pos = entry_widget.index("insert")
            last_end = None
            for m in sentence_end_re.finditer(s):
                if m.end() <= pos:
                    last_end = m.end()
                else:
                    break
            new_pos = last_end if last_end is not None else 0
            entry_widget.icursor(new_pos)

        def entry_move_to_sentence_end(entry_widget):
            s = entry_widget.get()
            pos = entry_widget.index("insert")
            found = None
            for m in sentence_end_re.finditer(s):
                if m.start() >= pos:
                    found = m.end()
                    break
            new_pos = len(s) if found is None else found
            entry_widget.icursor(new_pos)

        def on_key_up(event):
            if current_pos["pos"] > 0:
                prev_action()
            return "break"

        def on_key_down(event):
            if current_pos["pos"] < len(self.filtered_indices) - 1:
                next_action()
            return "break"

        popup_tree.bind("<ButtonRelease-1>", on_tree_click)
        popup_tree.bind("<<TreeviewSelect>>", on_tree_select)
        popup_tree.bind("<Up>", on_key_up)
        popup_tree.bind("<Down>", on_key_down)

        def editor_text_on_up(event):
            text_move_to_sentence_start(editor_question_text)
            return "break"

        def editor_text_on_down(event):
            text_move_to_sentence_end(editor_question_text)
            return "break"

        editor_question_text.bind("<Up>", editor_text_on_up)
        editor_question_text.bind("<Down>", editor_text_on_down)

        for ent in editor_entries.values():
            ent.bind("<Up>", lambda e, w=ent: (entry_move_to_sentence_start(w), "break"))
            ent.bind("<Down>", lambda e, w=ent: (entry_move_to_sentence_end(w), "break"))

        tk.Button(nav_frame, text="<< Previous", command=prev_action).pack(side="left", padx=6)
        tk.Button(nav_frame, text="Save", command=save_current).pack(side="left", padx=6)
        tk.Button(nav_frame, text="Next >>", command=next_action).pack(side="left", padx=6)
        tk.Button(nav_frame, text="Close", command=self.edit_popup.destroy).pack(side="right", padx=6)

        self.edit_popup.bind("<Control-s>", lambda e: save_current())

        refresh_popup_table()

        popup_search_entry.focus_set()

    def copy_selected_questions(self):
        selected_items = self.tree.selection()
        if not selected_items:
            messagebox.showwarning("Warning", "Please select one or more questions to copy.")
            return
        indices = [int(i) for i in selected_items]
        copied_rows = self.df.loc[indices].copy()
        if "Q no" in copied_rows.columns:
            copied_rows = copied_rows.drop(columns=["Q no"], errors="ignore")
        self.df = pd.concat([self.df, copied_rows], ignore_index=True)
        save_data(self.df, self.current_csv_file)
        self.refresh_table()
        messagebox.showinfo("Copied", f"Copied {len(copied_rows)} question(s) successfully!")

    def delete_question(self):
        selected_items = self.tree.selection()
        if not selected_items:
            messagebox.showwarning("Warning", "Select one or more questions to delete.")
            return
        if messagebox.askyesno("Confirm Delete", "Are you sure you want to delete the selected question(s)?"):
            indices = [int(i) for i in selected_items]
            self.df.drop(index=indices, inplace=True)
            self.df.reset_index(drop=True, inplace=True)
            save_data(self.df, self.current_csv_file)
            self.refresh_table()
            messagebox.showinfo("Deleted", f"Deleted {len(indices)} question(s) successfully!")

    def import_csv(self):
        file_path = filedialog.askopenfilename(title="Select CSV File", filetypes=[("CSV Files", "*.csv")])
        if file_path:
            new_df = pd.read_csv(file_path)
            new_df.columns = new_df.columns.str.strip()
            new_df = new_df.reindex(columns=[c for c in COLUMNS if c != "Q no"])
            self.df = pd.concat([self.df, new_df], ignore_index=True)
            save_data(self.df, self.current_csv_file)
            self.refresh_table()
            messagebox.showinfo("Imported", f"Imported {len(new_df)} questions successfully!")

    def export_csv(self):
        file_path = filedialog.asksaveasfilename(title="Save Exported CSV", defaultextension=".csv", filetypes=[("CSV Files", "*.csv")])
        if not file_path:
            return
        export_df = self.df.drop(columns=["Q no"], errors="ignore")
        export_df = export_df.reindex(columns=[c for c in COLUMNS if c != "Q no"])
        export_df.to_csv(file_path, index=False)
        messagebox.showinfo("Exported", f"Data exported successfully to:\n{file_path}")

    def normalize_text(self, text):
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r"[^a-z0-9+\-*/= ]", "", text)
        text = re.sub(r"\s*([+\-*/=])\s*", r"\1", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def remove_duplicates(self):
        normalized_questions = self.df["Question"].fillna("").apply(self.normalize_text)
        normalized_answers = self.df["Answer"].fillna("").apply(self.normalize_text)
        def normalize_options(row):
            opts = [self.normalize_text(row.get(f"Option{i}", "")) for i in range(1, 6)]
            opts = [o for o in opts if o]
            return "|".join(sorted(opts))
        normalized_options = self.df.apply(normalize_options, axis=1)
        fingerprint = normalized_questions + "||" + normalized_options + "||" + normalized_answers
        dup_mask = fingerprint.duplicated(keep="first")
        removed = int(dup_mask.sum())
        if removed == 0:
            messagebox.showinfo("No Duplicates", "No duplicate or similar questions found.")
            return
        self.df = self.df[~dup_mask].reset_index(drop=True)
        save_data(self.df, self.current_csv_file)
        self.refresh_table()
        messagebox.showinfo("Duplicates Removed", f"Removed {removed} similar/duplicate row(s).")

    def save_current_csv(self):
        """Save current in-memory DataFrame to the selected CSV file."""
        try:
            save_data(self.df, self.current_csv_file)
            messagebox.showinfo("Saved", f"Changes saved to:\n{self.current_csv_file}")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to save changes:\n{e}")

if __name__ == "__main__":
    root = tk.Tk()
    app = QuizManager(root)
    root.mainloop()

# GENERATOR
# TYPE: Basic
### STATUS: OK

In [ ]:
import pandas as pd
import random
from IPython.display import display, clear_output
import ipywidgets as widgets

csv_file = "January (378).csv"
# csv_file = "CORRECT.csv"
# csv_file = "quiz_data.csv"
# csv_file = "quiz_data_adda247.csv"
df = pd.read_csv(csv_file)

df.columns = df.columns.str.strip().str.lower()

question_candidates = [col for col in df.columns if 'question' in col]
answer_candidates = [col for col in df.columns if 'answer' in col]
option_cols = [col for col in df.columns if 'option' in col]

if not question_candidates or not answer_candidates or not option_cols:
    raise ValueError("❌ Could not find required columns (question, answer, optionX) in the CSV file.")

question_col = question_candidates[0]
answer_col = answer_candidates[0]

score = 0
attempted = 0
correct_count = 0
incorrect_count = 0
skipped_count = 0
selected_num_questions = None
questions_master = None
review_log = []

def start_screen():
    """Initial start screen with number of questions selector."""
    clear_output(wait=True)
    total_available = len(df)
    display(widgets.HTML(f"<h3>📊 Total Questions Available: {total_available}</h3>"))
    display(widgets.HTML("<h3>🎯 Select number of questions for this round:</h3>"))
    
    options = [10, 20, 30, 40, 50]
    dropdown = widgets.Dropdown(options=options, description="Questions:")
    start_btn = widgets.Button(description="Start Quiz", button_style='success')
    
    display(widgets.VBox([dropdown, start_btn]))
    
    def start(ev):
        global selected_num_questions
        selected_num_questions = dropdown.value
        start_quiz()
    
    start_btn.on_click(start)

def start_quiz():
    """Start or restart quiz using the chosen number of questions."""
    global score, attempted, correct_count, incorrect_count, skipped_count, questions_master, review_log
    score = 0
    attempted = 0
    correct_count = 0
    incorrect_count = 0
    skipped_count = 0
    review_log = []
    
    questions_master = df.sample(frac=1).reset_index(drop=True)
    
    if selected_num_questions and len(questions_master) > selected_num_questions:
        questions_master = questions_master.iloc[:selected_num_questions]
    
    if len(questions_master) == 0:
        clear_output(wait=True)
        display(widgets.HTML("<h3>❌ No questions available in the dataset.</h3>"))
        return
    
    show_question(0, questions_master)

def show_question(index, questions):
    """Display a question with options + skip button."""
    global attempted, skipped_count
    clear_output(wait=True)
    row = questions.iloc[index]
    
    display(widgets.HTML(f"<h3 style='font-weight:bold;'>Q{index+1}/{len(questions)}: {row[question_col]}</h3>"))
    
    options = [row[col] for col in option_cols if pd.notna(row[col])]
    random.shuffle(options)
    
    buttons = []
    skip_btn = widgets.Button(description="Skip Question", button_style='warning')
    display(skip_btn)

    def on_click(b):
        global score, attempted, correct_count, incorrect_count
        attempted += 1
        for btn in buttons:
            btn.disabled = True
        skip_btn.close()
        
        if b.description == row[answer_col]:
            score += 1
            correct_count += 1
            review_log.append(("Correct", row[question_col], row[answer_col]))
            display(widgets.HTML("<span style='color:green; font-size:20px;'>✅ Correct!</span>"))
        else:
            score -= 0.25
            incorrect_count += 1
            review_log.append(("Incorrect", row[question_col], row[answer_col]))
            display(widgets.HTML(f"<span style='color:red; font-size:20px;'>❌ Wrong! Correct answer: {row[answer_col]}</span>"))
        
        display(widgets.HTML(
            f"<p>Attempted: {attempted} | Correct: {correct_count} | Incorrect: {incorrect_count} | Skipped: {skipped_count} | Current Score: {score}</p>"
        ))
        
        show_control_buttons(index, questions)

    for option in options:
        btn = widgets.Button(description=str(option), layout=widgets.Layout(width='auto'))
        btn.on_click(on_click)
        buttons.append(btn)
        display(btn)

    def skip(ev):
        global skipped_count
        skipped_count += 1
        review_log.append(("Skipped", row[question_col], row[answer_col]))
        if index + 1 < len(questions):
            show_question(index + 1, questions)
        else:
            end_quiz()
    skip_btn.on_click(skip)

def show_control_buttons(index, questions):
    """Show restart, quit, and next buttons (after answering)."""
    restart_btn = widgets.Button(description="Restart Quiz", button_style='success')
    quit_btn = widgets.Button(description="Quit Quiz", button_style='danger')
    
    display(widgets.HBox([restart_btn, quit_btn]))
    
    def restart(ev):
        start_quiz()
    
    def quit(ev):
        global selected_num_questions
        selected_num_questions = None
        clear_output(wait=True)
        display(widgets.HTML("<h3>Quiz exited. Thanks for playing!</h3>"))
        show_review()
        start_screen()
    
    restart_btn.on_click(restart)
    quit_btn.on_click(quit)
    
    if index + 1 < len(questions):
        next_btn = widgets.Button(description="Next Question", button_style='info')
        display(next_btn)
        def next_q(ev):
            show_question(index + 1, questions)
        next_btn.on_click(next_q)
    else:
        end_quiz()

def end_quiz():
    """Show final quiz results with only restart and quit, plus review log."""
    clear_output(wait=True)
    display(widgets.HTML(
        f"<h3>🎉 Quiz finished!</h3>"
        f"<p>Attempted: {attempted} | Correct: {correct_count} | Incorrect: {incorrect_count} | Skipped: {skipped_count} | Final Score: {score}</p>"
    ))
    
    show_review()
    
    restart_btn = widgets.Button(description="Restart Quiz", button_style='success')
    quit_btn = widgets.Button(description="Quit Quiz", button_style='danger')
    display(widgets.HBox([restart_btn, quit_btn]))
    
    def restart(ev):
        start_quiz()
    
    def quit(ev):
        global selected_num_questions
        selected_num_questions = None
        clear_output(wait=True)
        display(widgets.HTML("<h3>Quiz exited. Thanks for playing!</h3>"))
        show_review()
        start_screen()
    
    restart_btn.on_click(restart)
    quit_btn.on_click(quit)

def show_review():
    """Show detailed review of all questions answered/skipped."""
    if not review_log:
        return
    
    display(widgets.HTML("<h3>📘 Review Section</h3>"))
    
    for status, question, answer in review_log:
        if status == "Correct":
            color = "green"
            icon = "✅"
        elif status == "Incorrect":
            color = "red"
            icon = "❌"
        else:
            color = "orange"
            icon = "⏭️"
        
        display(widgets.HTML(
            f"<p><b>{icon} [{status}]</b> <span style='color:{color};'>{question}</span><br>"
            f"<b>Answer:</b> {answer}</p><hr>"
        ))

start_screen()

## TYPE: With non repetative counter's for better efficiency
### STATUS: OK

In [ ]:
import pandas as pd
import random
import os
from IPython.display import display, clear_output
import ipywidgets as widgets

pd.set_option("display.max_colwidth", None)

# csv_file = "quiz_data.csv"
csv_file = "January (378).csv"
# csv_file = "quiz_data_adda247.csv"

question_col = None
answer_col = None
option_cols = []

CORRECT_FILE = "correct.csv"
# CORRECT_FILE = "correct_adda247.csv"

def load_filtered_questions():
    global question_col, answer_col, option_cols

    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.strip().str.lower()

    df['__original_csv_index__'] = df.index
    df['__original_csv_row__'] = df.index + 2

    question_candidates = [col for col in df.columns if 'question' in col]
    answer_candidates = [col for col in df.columns if 'answer' in col]
    option_cols = [col for col in df.columns if 'option' in col]

    if not question_candidates or not answer_candidates or not option_cols:
        raise ValueError("❌ Could not find required columns (question, answer, optionX) in the CSV file.")

    question_col = question_candidates[0]
    answer_col = answer_candidates[0]

    if os.path.exists(CORRECT_FILE):
        correct_df = pd.read_csv(CORRECT_FILE)
        correct_df.columns = correct_df.columns.str.strip().str.lower()

        if question_col in correct_df.columns:
            df = df[~df[question_col].isin(correct_df[question_col])]
            df = df.reset_index(drop=True)

    return df

def append_to_csv(filename, row):
    """Append question to CSV if not already present."""
    columns = [question_col] + option_cols + [answer_col]

    if os.path.exists(filename):
        df_existing = pd.read_csv(filename)
        df_existing.columns = df_existing.columns.str.strip().str.lower()
    else:
        df_existing = pd.DataFrame(columns=columns)

    if not ((df_existing[question_col] == row[question_col]).any()):
        new_row = pd.DataFrame([[row[question_col]] +
                                [row[col] if col in row else None for col in option_cols] +
                                [row[answer_col]]], columns=columns)
        df_existing = pd.concat([df_existing, new_row], ignore_index=True)
        df_existing.to_csv(filename, index=False)

score = 0
attempted = 0
correct_count = 0
skipped_count = 0
selected_num_questions = None
questions_master = None
review_log = []
marked_errors = []
df = load_filtered_questions()

def start_screen():
    global df
    df = load_filtered_questions()
    clear_output(wait=True)

    if df.empty:
        display(widgets.HTML("<h3>🎉 You've answered all questions correctly already! Nothing new to quiz on.</h3>"))
        return

    total_available = len(df)
    display(widgets.HTML(f"<h3>📊 Total New Questions Available: {total_available}</h3>"))
    display(widgets.HTML("<h3>🎯 Select number of questions for this round:</h3>"))

    options = [n for n in [10, 20, 30, 40, 50] if n <= total_available]
    if total_available not in options:
        options.append(total_available)
    dropdown = widgets.Dropdown(options=options, description="Questions:")
    start_btn = widgets.Button(description="Start Quiz", button_style='success')
    display(widgets.VBox([dropdown, start_btn]))

    def start(ev):
        global selected_num_questions
        selected_num_questions = dropdown.value
        start_quiz()

    start_btn.on_click(start)

def start_quiz():
    global score, attempted, correct_count, skipped_count, questions_master, review_log, marked_errors
    score = 0
    attempted = 0
    correct_count = 0
    skipped_count = 0
    review_log = []
    marked_errors = []

    if df.empty:
        clear_output(wait=True)
        display(widgets.HTML("<h3>✅ No new questions left to practice.</h3>"))
        return

    questions_master = df.sample(frac=1).reset_index(drop=True)

    if selected_num_questions and len(questions_master) > selected_num_questions:
        questions_master = questions_master.iloc[:selected_num_questions]

    if len(questions_master) == 0:
        clear_output(wait=True)
        display(widgets.HTML("<h3>❌ No questions available in the dataset.</h3>"))
        return

    show_question(0, questions_master)

def show_question(index, questions):
    global attempted, skipped_count
    clear_output(wait=True)
    row = questions.iloc[index]

    display(widgets.HTML(f"<h3 style='font-weight:bold;'>Q{index+1}/{len(questions)}: {row[question_col]}</h3>"))

    options = [row[col] for col in option_cols if pd.notna(row[col])]
    random.shuffle(options)

    buttons = []
    skip_btn = widgets.Button(description="Skip Question", button_style='warning')
    mark_error_btn = widgets.Button(description="Mark as Error ⚠️", button_style='danger')
    display(widgets.HBox([skip_btn, mark_error_btn]))

    def on_click(b):
        global score, attempted, correct_count
        attempted += 1
        for btn in buttons:
            btn.disabled = True
        skip_btn.close()
        mark_error_btn.close()

        if b.description == row[answer_col]:
            score += 1
            correct_count += 1
            review_log.append(("Correct", row[question_col], row[answer_col]))
            append_to_csv(CORRECT_FILE, row)
            display(widgets.HTML("<span style='color:green; font-size:20px;'>✅ Correct!</span>"))
        else:
            score -= 0.25
            review_log.append(("Incorrect", row[question_col], row[answer_col]))
            display(widgets.HTML(f"<span style='color:red; font-size:20px;'>❌ Wrong! Correct answer: {row[answer_col]}</span>"))

        display(widgets.HTML(
            f"<p>Attempted: {attempted} | Correct: {correct_count} | Skipped: {skipped_count} | Current Score: {score}</p>"
        ))

        show_control_buttons(index, questions)

    for option in options:
        btn = widgets.Button(description=str(option), layout=widgets.Layout(width='auto'))
        btn.on_click(on_click)
        buttons.append(btn)
        display(btn)

    def skip(ev):
        global skipped_count, attempted
        skipped_count += 1
        attempted += 1
        review_log.append(("Skipped", row[question_col], row[answer_col]))
        if index + 1 < len(questions):
            show_question(index + 1, questions)
        else:
            end_quiz()

    skip_btn.on_click(skip)

    def mark_error(ev):
        global marked_errors
        original_csv_row = int(row['__original_csv_row__'])
        question_text = row[question_col]
        if (original_csv_row, question_text) not in marked_errors:
            marked_errors.append((original_csv_row, question_text))
        display(widgets.HTML("<p style='color:orange;'>⚠️ Marked this question for review later.</p>"))

    mark_error_btn.on_click(mark_error)

def show_control_buttons(index, questions):
    restart_btn = widgets.Button(description="Restart Quiz", button_style='success')
    quit_btn = widgets.Button(description="Quit Quiz", button_style='danger')
    display(widgets.HBox([restart_btn, quit_btn]))

    def restart(ev):
        global df
        df = load_filtered_questions()
        start_quiz()

    def quit(ev):
        global selected_num_questions, df
        selected_num_questions = None
        df = load_filtered_questions()
        clear_output(wait=True)
        display(widgets.HTML("<h3>Quiz exited. Thanks for playing!</h3>"))
        show_review()
        show_marked_errors()
        start_screen()

    restart_btn.on_click(restart)
    quit_btn.on_click(quit)

    if index + 1 < len(questions):
        next_btn = widgets.Button(description="Next Question", button_style='info')
        display(next_btn)

        def next_q(ev):
            show_question(index + 1, questions)

        next_btn.on_click(next_q)
    else:
        end_quiz()

def end_quiz():
    global df
    df = load_filtered_questions()
    clear_output(wait=True)
    display(widgets.HTML(
        f"<h3>🎉 Quiz finished!</h3>"
        f"<p>Attempted: {attempted} | Correct: {correct_count} | Skipped: {skipped_count} | Final Score: {score}</p>"
    ))

    show_review()
    show_marked_errors()

    restart_btn = widgets.Button(description="Restart Quiz", button_style='success')
    quit_btn = widgets.Button(description="Quit Quiz", button_style='danger')
    display(widgets.HBox([restart_btn, quit_btn]))

    def restart(ev):
        start_quiz()

    def quit(ev):
        global selected_num_questions, df
        selected_num_questions = None
        df = load_filtered_questions()
        clear_output(wait=True)
        display(widgets.HTML("<h3>Quiz exited. Thanks for playing!</h3>"))
        show_review()
        show_marked_errors()
        start_screen()

    restart_btn.on_click(restart)
    quit_btn.on_click(quit)

def show_review():
    if not review_log:
        return

    display(widgets.HTML("<h3>📘 Review Section (Full Summary)</h3>"))

    review_df = pd.DataFrame(review_log, columns=["Status", "Question", "Answer"])
    review_df = review_df[["Question", "Answer", "Status"]]
    review_df.index = review_df.index + 1
    review_df.index.name = "No."

    styled_review = review_df.style.set_properties(**{'text-align': 'left'})
    styled_review = styled_review.set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'left')]}]
    )

    display(styled_review)

def show_marked_errors():
    if not marked_errors:
        return

    display(widgets.HTML("<h3>⚠️ Marked Questions (CSV Reference)</h3>"))
#     display(widgets.HTML("<p>Row number = CSV line number (1-based, header = row 1)</p>"))

    marked_df = pd.DataFrame(marked_errors, columns=["CSV Row Number", "Question"])
    marked_df.index = marked_df.index + 1
    marked_df.index.name = "No."

    styled_marked = marked_df.style.set_properties(**{'text-align': 'left'})
    styled_marked = styled_marked.set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'left')]}]
    )

    display(styled_marked)

start_screen()

# TESTING